# Lab 3ii: Financial Analysis with Code Interpreter

## Overview

Add code execution capabilities to the Product Launch Agent for financial calculations, data analysis, and ROI modeling.

### Why Code Interpreter for Product Launch?

**Use Cases:**
- Calculate loan amortization schedules
- Analyze market data and trends
- Model pricing strategies and ROI
- Generate financial projections

### Prerequisites
- Python 3.11+
- AWS account with AgentCore Code Interpreter access
- Bedrock model access (Claude 4.5 Haiku)

## Step 1: Install Dependencies

In [ ]:
!pip install -U bedrock-agentcore strands-agents -q

## Step 2: Setup Code Interpreter Tool

In [ ]:
from bedrock_agentcore.tools.code_interpreter_client import code_session
from strands import Agent, tool
from strands.models import BedrockModel
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

@tool
def execute_financial_calculation(python_code: str) -> str:
    """
    Execute Python code for financial calculations and analysis.
    
    Args:
        python_code: Python code to execute (calculations, data analysis, modeling)
    
    Returns:
        Execution results including output and any errors
    """
    with code_session(region=region) as session:
        result = session.execute_code(python_code)
        
        if result.get('isError'):
            error_msg = result.get('content', [{}])[0].get('text', 'Unknown error')
            return f"Error: {error_msg}"
        
        # Extract stdout from structured content
        structured = result.get('structuredContent', {})
        stdout = structured.get('stdout', '')
        stderr = structured.get('stderr', '')
        
        if stderr:
            return f"Output: {stdout}\nWarnings: {stderr}"
        
        return stdout or "Code executed successfully (no output)"

print("✅ Code interpreter tool ready")

## Step 3: Create Financial Analysis Agent

In [ ]:
SYSTEM_PROMPT = """You are a financial product launch analyst with code execution capabilities.

Use execute_financial_calculation to:
- Calculate loan payments, amortization schedules
- Analyze pricing strategies and break-even points
- Model ROI and revenue projections
- Perform statistical analysis on market data

Always show your calculations with code for transparency."""

model = BedrockModel(
    model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    temperature=0.3,
    region_name=region
)

agent = Agent(
    model=model,
    tools=[execute_financial_calculation],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Financial analysis agent created")

## Step 4: Test Financial Calculations

### Test 1: Auto Loan Payment Calculator

In [ ]:
response = agent("Calculate monthly payment for a $30,000 auto loan at 5.99% APR for 60 months")
print(response)

### Test 2: Pricing Strategy Analysis

In [ ]:
response = agent(
    """Analyze pricing strategy for a credit card:
    - Annual fee: $95
    - Average balance: $5,000
    - APR: 18.99%
    - Rewards: 2% cashback
    Calculate annual revenue per customer and break-even point"""
)
print(response)

### Test 3: Market Data Analysis

In [ ]:
response = agent(
    """Analyze competitor rates and calculate our competitive position:
    Competitor rates: [6.25%, 5.75%, 6.10%, 5.90%, 6.00%]
    Our proposed rate: 5.99%
    Calculate mean, median, and our percentile ranking"""
)
print(response)

### Test 4: ROI Projection

In [ ]:
response = agent(
    """Project 3-year ROI for launching a new savings account:
    - Launch cost: $500,000
    - Expected customers Year 1: 10,000, Year 2: 25,000, Year 3: 50,000
    - Average deposit: $5,000
    - Net interest margin: 2.5%
    Calculate cumulative revenue and ROI by year"""
)
print(response)

## 🎉 Lab 3ii Complete!

You've successfully:
- Added code execution to the Product Launch Agent
- Performed financial calculations programmatically
- Analyzed pricing strategies and market data
- Generated ROI projections

**Next:** Integrate code interpreter with the full product launch agent in Lab 4.